# 05 - Stage 1 Lexicon Features

This notebook adds the project's existing lexicon files as a feature ablation. It compares the text-only baseline with TF-IDF plus normalized lexicon-category counts. It uses the same grouped repeated cross-validation protocol as notebook 04.

Lexicons are weak signals, not hand-written rules or the final detector. No message-level manual labels are created.

In [1]:
from pathlib import Path
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline, FeatureUnion

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
LEXICON_DIR = WORKSPACE / "lexicons"
MODEL_DIR = WORKSPACE / "models"
REPORT_DIR = WORKSPACE / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

messages_path = PROCESSED_DIR / "coded_stage_messages.csv"
if not messages_path.exists():
    raise FileNotFoundError("Run 01_coded_chat_data_loading.ipynb first.")

messages = pd.read_csv(messages_path, dtype={"category": "string"})
messages["text"] = messages["text"].fillna("")
messages["category"] = messages["category"].astype(str)

assert messages["category"].isin({"200", "600", "900"}).all()
assert not messages.duplicated(["chat_id", "msg_idx"]).any()

messages = messages.sort_values(["chat_id", "msg_idx"]).copy()
messages["previous_text"] = messages.groupby("chat_id")["text"].shift(1).fillna("")
messages["next_text"] = messages.groupby("chat_id")["text"].shift(-1).fillna("")
messages["model_text"] = (
    "previous: " + messages["previous_text"] + " target: " + messages["text"] + " next: " + messages["next_text"]
)

print("Messages:", len(messages), "Chats:", messages["chat_id"].nunique())

Messages: 10831 Chats: 56


## Lexicon loading

Each `lexicons/*.txt` file becomes one feature category. Leading comments and blank lines are ignored. Multi-word entries are matched after lowercasing and whitespace normalization. Counts are also reported as ratios to message length.

In [2]:
WORD_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")


def load_lexicon_file(path: Path) -> list[str]:
    entries = []
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip().lower()
        if line and not line.startswith("#"):
            entries.append(line)
    return entries


lexicons = {
    path.stem: load_lexicon_file(path)
    for path in sorted(LEXICON_DIR.glob("*.txt"))
}
lexicon_inventory = pd.DataFrame({
    "category": list(lexicons.keys()),
    "entry_count": [len(entries) for entries in lexicons.values()],
})

class LexiconTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lexicon_categories):
        self.lexicon_categories = lexicon_categories

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        feature_rows = []
        for text in X:
            normalized = re.sub(r"\s+", " ", str(text).lower()).strip()
            token_count = max(1, len(WORD_PATTERN.findall(normalized)))
            row = []
            padded = f" {normalized} "
            for entries in self.lexicon_categories.values():
                count = sum(padded.count(f" {entry.strip().lower()} ") for entry in entries if entry.strip())
                row.extend([count, count / token_count])
            feature_rows.append(row)
        return np.asarray(feature_rows, dtype=float)


display(lexicon_inventory)
display("Lexicon categories loaded:", len(lexicons))

,category,entry_count
0,actionV,10
1,actN,13
2,approachN,9
3,approachV,9
4,commAdj,42
5,commN,156
6,commV,107
7,commWord,1020
8,familyN,10
9,infoN,13


'Lexicon categories loaded:'

17

## Incremental comparison

The word and character TF-IDF baseline is trained again, followed by the same text features with lexicon count and normalized-rate features. Models use the same folds for a fair paired comparison.

In [3]:
def text_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 3), min_df=2, sublinear_tf=True)),
        ("character", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ])


def classifier():
    return LogisticRegression(
        class_weight="balanced",
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_SEED,
    )


def build_model(include_lexicon: bool) -> Pipeline:
    branches = [("text", text_features())]
    if include_lexicon:
        branches.append(("lexicon", LexiconTransformer(lexicons)))
    return Pipeline([
        ("features", FeatureUnion(branches)),
        ("classifier", classifier()),
    ])


X = messages["model_text"].to_numpy()
y = messages["category"].to_numpy()
groups = messages["chat_id"].to_numpy()

n_splits = min(5, messages["chat_id"].nunique())
all_rows = []

for include_lexicon, model_name in [(False, "text_only"), (True, "text_plus_lexicon")]:
    for repeat in range(3):
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED + repeat)
        for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups)):
            model = build_model(include_lexicon)
            model.fit(X[train_index], y[train_index])
            predictions = model.predict(X[test_index])
            precision, recall, f1, _ = precision_recall_fscore_support(
                y[test_index], predictions,
                labels=["200", "600", "900"],
                average="macro",
                zero_division=0,
            )
            all_rows.append({
                "model": model_name,
                "repeat": repeat,
                "fold": fold,
                "macro_precision": precision,
                "macro_recall": recall,
                "macro_f1": f1,
            })

metrics = pd.DataFrame(all_rows)
summary = metrics.groupby("model", sort=False)[["macro_precision", "macro_recall", "macro_f1"]].agg(["mean", "std"])

display(metrics)
display(summary)

,model,repeat,fold,macro_precision,macro_recall,macro_f1
0,text_only,0,0,0.649840,0.657524,0.647942
1,text_only,0,1,0.718501,0.696624,0.704893
2,text_only,0,2,0.732212,0.738773,0.732304
3,text_only,0,3,0.724413,0.695299,0.704985
4,text_only,0,4,0.678275,0.679753,0.675762
5,text_only,1,0,0.649840,0.657524,0.647942
6,text_only,1,1,0.718501,0.696624,0.704893
7,text_only,1,2,0.732212,0.738773,0.732304
8,text_only,1,3,0.724413,0.695299,0.704985
9,text_only,1,4,0.678275,0.679753,0.675762


macro_precision           macro_recall            macro_f1  \
                             mean       std         mean       std      mean   
model                                                                          
text_only                0.700648  0.032607     0.693595  0.027565  0.693177   
text_plus_lexicon        0.709021  0.032097     0.711251  0.027962  0.706785   

                             
                        std  
model                        
text_only          0.029846  
text_plus_lexicon  0.030794

## Save the lexicon model and results

The complete text-and-lexicon model is fitted only after grouped comparison. Use the saved results with the baseline notebook rather than selecting lexicon features after observing test fold results.

In [4]:
lexicon_model = build_model(True)
lexicon_model.fit(X, y)
joblib.dump(lexicon_model, MODEL_DIR / "stage1_tfidf_lexicon_logistic.joblib")

metrics.to_csv(REPORT_DIR / "stage1_lexicon_fold_metrics.csv", index=False)
summary.to_csv(REPORT_DIR / "stage1_lexicon_metric_summary.csv")
lexicon_inventory.to_csv(REPORT_DIR / "stage1_lexicon_inventory.csv", index=False)

print("Saved lexicon model and Stage 1 ablation results.")

Saved lexicon model and Stage 1 ablation results.
